In [1]:
from posydon.utils import constants as const
import numpy as np

In [2]:
# CHECKED and WORKING 

def orbital_separation_from_period(Mass1, Mass2, period_days):
    """
    Calculate the orbital separation "a" given the period of the system from the 3rd Kepler's law. 
    
    Parameters 
    ---------
    Mass1 : float 
        Mass of the central body/accretor where Mass1 > Mass2 in [M⊙]
    Mass2 : float
        Mass of the orbiting object/donor where Mass2 < Mass1 in [M⊙]
    period_days : float 
        Period of the orbiting companion in [days]
    
    Return
    ------
    a : float 
        Orbital separation of the binary system in [cm]
    """
    period_seconds = period_days * 24 * 3600 # * const.day2sec
    M1_gr = Mass1 * const.Msun
    M2_gr = Mass2 * const.Msun
    a = (const.standard_cgrav * (M1_gr + M2_gr) * period_seconds**2 /(4 * np.pi**2))**(1/3) # [cm]
    
    return a

In [12]:
a = orbital_separation_from_period(1.0, 0.000003, 365.25)
print(f"The orbital separation is {a/1e5} km.")

The orbital separation is 149615799.5221468 km.


In [3]:
# CHECKED and WORKING

def Get_velocities(Macc, M_donor_Msun, R_donor_Rsun, orbital_period_days, scale = 2.6):
    """
    Compute different velocities : 
        - Relative velocity of a wind emitted form a companion star including orbital + esacpe velocity of the companion
        - Orbital velocity of the companion/donor star
        - Wind velocity if we suppose that the required velocity to be unbound is > escape velocity

    Parameters
    ----------
    Macc : float
        Mass of the accretor in [M⊙]
    M_donor_Msun : float
        Donor mass given in [M⊙] 
    R_donor_Rsun : float
        Donor radius given in [R⊙] 
    scale : float 
        Scale parameter to rescale the wind speed from the donor
        
    Return
    ------
    V_rel : float
        Relative velocitiy accoring to the central object/accretor [cm/s] 
    V_orb : float
        Orbital velocity of the donor star [cm/s]
    V_wind : float
        Wind velocity from the donor to the accretor [cm/s]
    """

    M_donor_gr = M_donor_Msun * const.Msun # [gr]
    R_donor_cm = R_donor_Rsun * const.Rsun # [cm]
    a_cm = orbital_separation_from_period(Macc, M_donor_Msun, orbital_period_days) #[cm]

    # Escape and wind velocities
    V_esc = np.sqrt(2 * const.standard_cgrav * M_donor_gr / max(R_donor_cm, 1e-10)) # [cm/s]
    V_wind = scale * V_esc  # [cm/s] rescaling

    # Orbital velocity 
    V_orb = 2 * np.pi * a_cm / (float(orbital_period_days) * const.day2sec) # [cm/s]
    # Relative velocity 
    V_rel = np.sqrt(V_wind ** 2 + V_orb ** 2) # [cm/s]

    return V_rel, V_orb, V_wind

In [11]:
V_rel, V_orb, V_wind = Get_velocities(1.0, 0.000003, 0.0091677, 365.25, 1.0)

print("V_rel = ", V_rel/1e5, " km/s")
print("V_orb = ", V_orb/1e5, " km/s")
print("V_wind = ", V_wind/1e5, " km/s")

V_rel =  31.81541622105164  km/s
V_orb =  29.7888240322293  km/s
V_wind =  11.173480751119316  km/s


In [25]:
# CHECK and WORKING

def BH_efficiency(mass, spin, dimensional_spin = True): 
    """
    Compute the radiative efficiency of a Kerr black hole (or Schwarzchild BH if spin = 0)
    as a function of the spin parameter a and the mass according to Novikov-Thorne radiative efficiency theory.

    Source:
            Bambi, C., Malafarina, D., & Tsukamoto, N. (2014).
            "Note on the radiation emitted by a thin accretion disk around a rotating compact object."
            *Phys. Rev. D*, 89(12), 127302. https://arxiv.org/abs/1406.2181
    
    Parameters
    ---------- 
    mass : float
        mass of the central black hole in [M⊙]
    spin : float
        Spin of the black hole. Can be given as dimensionless spin "a*" or dimensional spin "a" in [cm].
        Its value can be given either positively (prograde accretion) or negatively (retrograde accretion) 
        (spin  ∈ [-0.998, 0.998] with ± 0.998 being the Thorne limit).
        
    dimensional_spin : bool, optional 
        If True, convert dimensional spin "a" to dimentionless spin "a*".

    Returns
    -------
    tuple
        A tuple with three floats: (E_ISCO, r_tilde, eta).
        - E_ISCO (float): Specific energy at the ISCO, in units of m c^2.
        - r_tilde (float): Radius of the ISCO in units of G M / c^2.
        - eta (float): Radiative efficiency (1 - E_ISCO), dimensionless.
    """
   
    if  mass < 0 : 
        raise ValueError("Mass must be positive")

    M = mass # * const.Msun # [gr]

    # Conversion to dimensionless spin
    if dimensional_spin:
        a_star = const.clight * spin/(const.standard_cgrav * M)
    else: 
        a_star = float(spin)
    
    # To control that the spin is in the range that gives the maximal efficiency
    a_star = float(np.clip(a_star, -0.998, 0.998))

    #=== Compute efficiency === 
    
    # Use sign = +1 for spin ≥ 0 (prograde) and -1 otherwise (retrograde)
    sign = 1 if a_star >= 0 else -1

    Z1 = 1 + (1 - a_star**2)**(1/3) * ((1 + a_star)**(1/3) + (1 - a_star)**(1/3))
    Z2 = np.sqrt(3 * a_star**2 + Z1**2)

    # Radius at the innermost stable circular orbit (ISCO)
    r_tilde = 3 + Z2 - sign * np.sqrt((3 - Z1)*(3 + Z1 + 2 * Z2)) # Dimensionless
    
    # Energy at ISCO in units of mc^2 (i.e E_ISCO < 1, E = 1 means particle energy at infinity)
    E_ISCO = (r_tilde**(3/2) - 2 * r_tilde**(1/2) + a_star)/(r_tilde**(3/4) * np.sqrt(r_tilde**(3/2) - 3 * r_tilde**(1/2) + 2 * a_star))
    
    # Radiative efficiency
    eta = 1 - E_ISCO
    return E_ISCO, r_tilde, eta

In [26]:
E_ISCO, r_tilde, eta = BH_efficiency(10, 0.0, dimensional_spin = False)
print(f"Energy at innermost stable circular orbit (ISCO) is {E_ISCO:.5f}.")
print(f"Radius at ISCO in units of GM/c^2 is {r_tilde:.2f} M⊙.")
print(f"Radiative efficiency is {eta:.5f}.")

# For spin increasing from 0.0, the radius at ISCO decreases. For spin decreasing from 0.0, the radius at ISCO increases.
# E is at maxium = 1, i.e. the energy of a particle at infinity in mc^2 units. 

Energy at innermost stable circular orbit (ISCO) is 0.94281.
Radius at ISCO in units of GM/c^2 is 6.00 M⊙.
Radiative efficiency is 0.05719.


In [24]:
def BHL_mdot_acc(M_acc_Msun, Mdot_w_Msun_per_yr, a_cm, v_rel_cms, alpha_BHL=1.0):
    """
    Calculate the Bondi-Hoyle Littleton accretion rate in gram per second.

    Parameters
    ----------
    M_acc_Msun : float 
        Mass of the accretor in [M⊙]
    Mdot_w_Msun_per_yr : float
        Accretion rate from winds in [M⊙/yr]
    a_cm : float
        Orbital separation of the binary system in [cm]
    v_rel_cms : float
        Relative velocity of the accreted material in [cm/s]
        This relative velocity is a combination of the wind speed and orbital velocity
    alpha_BHL : float
        Alpha parameter of Bondi-Hoyle Littleton. Initially set at 1 if not given by user
        Can be changed accroding to the model used and alpha_BHL ∈[1,2]
    
    Return
    ------
    Mdot_w : float 
        Accretion rate through winds in [gr/s]  
    """

    Macc = float(M_acc_Msun) * const.Msun # [gr]
    
    # Accreted material from winds
    Mdot_w_gr_per_sec = float(max(Mdot_w_Msun_per_yr, 0.0)) * const.Msun / const.secyer  # [g/s]
    Vrel_power_4 = max(v_rel_cms, 1.0)**4
    Sep_power_2 = max(a_cm, 1.0)**2

    frac = alpha_BHL * (const.standard_cgrav**2 * Macc**2) / (Sep_power_2 * Vrel_power_4)
    return frac * Mdot_w_gr_per_sec  # g/s

In [ ]:
# CORRECTLY DEFINED, NOT TESTED

def Compute_Lx(donor_mass, donor_log_R, accretor_mass, accretor_radius, accretor_type, 
               donor_log_mdot, log_Mtransfer_rate, accretor_log_mdot,
               H1_content_donor, orbital_period_days = 15, accretor_spin  = 0.0): 
    """
    Compute the X-ray luminosity of a binary system as long as there is a compact accretor
    
    Source : Misra & al., A&A 672, A99(2023)
    
    Parameters
    ----------
    donor_mass : float
        Mass of the donor star in [M⊙]
    donor_log_R : float
        Log10(Radius of the donor star) in [R⊙]
    accretor_mass : float
        Mass of the accretor in [M⊙]
    accretor_radius : float
        Radius of the accretor in [R⊙]
    accretor_type : str
        Type of the accretor. Can be either 'NS' or 'BH'
    donor_log_mdot : float
        Log10(Mass loss rate from the donor star) in [M⊙/yr]. Is the total mass loss rate from the donor star including both RLO and winds.
    log_Mtransfer_rate : float
        Log10(Mass transfer rate) in [M⊙/yr]. A fraction of donor_log_mdot.
    accretor_log_mdot : float
        Log10(Mass accretion rate onto the accretor) in [M⊙/yr]. Is the total mass accretion rate onto the accretor including both RLO and winds.
    H1_content_donor : float
        Hydrogen mass fraction of the donor star's surface (surface_h1_f).
    orbital_period_days : float
        Orbital period of the binary system in [days] initially set a 15 days. If not given, wind accretion cannot be computed.
    accretor_spin : float
        Spin of the accretor in the case of BH accretor. Initially set at 0.0 (non spinning BH (Schwarzchild)).

    Returns
    -------
    tuple
        A tuple with three floats: (Lx_total, Lx_RLO, Lx_BHL).
        - Lx_total (float): Total X-ray luminosity of the binary system in [erg/s].
        - Lx_RLO (float): X-ray luminosity from Roche lobe overflow in [erg/s].
        - Lx_BHL (float): X-ray luminosity from Bondi-Hoyle-Littleton wind accretion in [erg/s].
    """

    #===== Efficiency according to CO ===== 
    if accretor_type == 'NS':
        eta = const.standard_cgrav * accretor_mass /(accretor_radius * const.clight**2) # ~0.2 for NS
    elif accretor_type == 'BH':
        eta = BH_efficiency(accretor_mass, accretor_spin)
    else:
        return np.nan, np.nan, np.nan # Non compact accretor case
    
    #===== Rates RLO/Winds ======
    Mdot_RLO_grsec = 10**accretor_log_mdot * const.Msun / const.secyer # RLO only in [g/s]
    mdot_Wind_Only_Msunyr = max(10**donor_log_mdot - 10**log_Mtransfer_rate, 0.0) # Only mass loss through winds in [M⊙/yr]
    Mdot_BHL_grsec = BHL_mdot_acc(accretor_mass, mdot_Wind_Only_Msunyr, a, V_rel) # in [g/s]

    #===== Required parameters =====
    a = orbital_separation_from_period(orbital_period_days, donor_mass, accretor_mass) * const.Rsun # In [cm]
    Donor_Rsun = 10**donor_log_R * const.Rsun # In [cm]
    V_rel, _, _ = Get_velocities(accretor_mass, donor_mass, Donor_Rsun, orbital_period_days) # In [cm/s]
    
    #===== X-ray luminosity computation =====
    Mdot_Edd = 4 * np.pi * const.standard_cgrav * accretor_mass * const.Msun / (0.2 * (1 + H1_content_donor) * const.clight * eta) # In [g/s]
    Mdot_Edd_Msunyr = Mdot_Edd * const.secyer / const.Msun # In [M⊙/yr]
    # Eddington ratio
    m_dot =  10**log_Mtransfer_rate / Mdot_Edd_Msunyr

    # Sub-Eddington accretion
    if m_dot < 1.0 : 

        Lx_RLO = eta * Mdot_RLO_grsec * const.clight**2 if Mdot_RLO_grsec > 0.0 else 0.0 # [erg/s]
        
        if np.isfinite(Mdot_BHL_grsec) and Mdot_BHL_grsec > 0.0: 
            Lx_BHL = eta * Mdot_BHL_grsec * const.clight**2 # [erg/s]
        else:
            Lx_BHL = 0.0 # [erg/s]

        Lx_total = Lx_RLO + Lx_BHL 

    # Super-Eddington accretion
    else: 

        L_edd = 4 * np.pi * const.standard_cgrav * accretor_mass * const.Msun * const.clight / (0.2 * (1 + H1_content_donor)) # [erg/s]

        # Inclusion of beaming factor if super-Eddington accretion
        Lx_RLO = L_edd * m_dot**2 / 73 * (1 + np.log(m_dot)) if m_dot > 8.5 else L_edd * (1 + np.log(m_dot)) # RLO luminosity [erg/s]
        Lx_BHL = L_edd * m_dot**2 / 73 * (1 + np.log(m_dot)) if m_dot > 8.5 else L_edd * (1 + np.log(m_dot)) # BHL luminosity [erg/s]

        Lx_total = Lx_RLO + Lx_BHL 
    

    return float(Lx_total), float(Lx_RLO), float(Lx_BHL)

In [27]:
def XRB_selection_function(history_chunk, oneline_chunk, formation_channels_chunk = None, **kwargs):
    '''
    A XRB selection function to create a population of XRBs, where we store some important information.
    This function identifies X-ray binaries as an accretor and a donor star.

    Parameters
    ----------
    history_chunk : pd.DataFrame
        A chunk of the history DataFrame from population class binaries. 
        Specific lines need to be provided. 
    oneline_chunk : pd.DataFrame
        A chunk of the oneline DataFrame from population class binaries.
    formation_channels_chunk : pd.DataFrame, optional
        A chunk of the formation channels DataFrame of the binaries. Default is None.
    **kwargs : dict, optional
        Additional arguments passed to Compute_Lx function.

    Returns
    -------
    DF_XRBs : pd.DataFrame
        A DataFrame containing the selected XRBs with accretor/donor data and XLFs associated to each binary.
    
    '''
    #===== Check inputs =====
    required = [
        'S1_state_f','S2_state_f','S1_mass_f','S2_mass_f',
        'S1_log_R_f','S2_log_R_f','S1_lg_mdot_f','S2_lg_mdot_f',
        'S1_surf_avg_omega_div_omega_crit_f','S2_surf_avg_omega_div_omega_crit_f',
        'S1_spin_f','S2_spin_f','S1_surface_h1_f','S2_surface_h1_f',
        'time_f','time_i','metallicity','orbital_period_f','eccentricity_f','lg_mtransfer_rate_f'
    ]

    missing = [col for col in required if col not in oneline_chunk.columns]
    if missing:
        raise KeyError(f"oneline_chunk missing required columns: {missing}")

    #===== Initialization =====
    # Create an empty Dataframe with indices of the selected binaries
    indices = oneline_chunk.index.to_numpy()
    DF_XRBs = pd.DataFrame(index = indices)

    DF_XRBs['time'] = oneline_chunk['time_f'] * 1e-6 # Myr
    DF_XRBs['time_of_birth'] = oneline_chunk['time_i'] * 1e-6 # Myr
    
    # Compact objects types for filtering
    Compact_types = {'NS','BH'}

    #===== Optional parameters **kwargs =====
    scale = kwargs.get('scale', 2.6) # Scale parameter to rescale
    alpha_bhl = kwargs.get('alpha_bhl', 1.0) # Alpha parameter of Bondi-Hoyle Littleton

    #===== Identify accretor/donor and their properties =====
    Star1 = oneline_chunk['S1_state_f']
    Star2 = oneline_chunk['S2_state_f']

    mask1 = (Star1.isin(Compact_types)) & (~Star2.isin(Compact_types))
    mask2 = (Star2.isin(Compact_types)) & (~Star1.isin(Compact_types))

    # Assignement of accretor/donor and their properties
    DF_XRBs['Accretor mass'] = np.where(mask1, oneline_chunk['S1_mass_f'], np.where(mask2, oneline_chunk['S2_mass_f'], np.nan))

    DF_XRBs['Accretor state'] = np.where(mask1, oneline_chunk['S1_state_f'], np.where(mask2, oneline_chunk['S2_state_f'], None))

    DF_XRBs['Accretor log(R)'] = np.where(mask1, oneline_chunk['S1_log_R_f'], np.where(mask2, oneline_chunk['S2_log_R_f'], np.nan))

    DF_XRBs['Accretor log(mdot)'] = np.where(mask1, oneline_chunk['S1_lg_mdot_f'],  np.where(mask2, oneline_chunk['S2_lg_mdot_f'], np.nan))

    DF_XRBs['Accretor spin'] = np.where(mask1, oneline_chunk['S1_spin_f'], np.where(mask2, oneline_chunk['S2_spin_f'], np.nan))

    DF_XRBs['Donor mass'] = np.where(mask1, oneline_chunk['S2_mass_f'], np.where(mask2, oneline_chunk['S1_mass_f'], np.nan))

    DF_XRBs['Donor state'] = np.where(mask1, oneline_chunk['S2_state_f'], np.where(mask2, oneline_chunk['S1_state_f'], None))

    DF_XRBs['Donor log(R)'] = np.where(mask1, oneline_chunk['S2_log_R_f'], np.where(mask2, oneline_chunk['S1_log_R_f'], np.nan))

    DF_XRBs['Donor log(mdot)'] = np.where(mask1, oneline_chunk['S2_lg_mdot_f'], np.where(mask2, oneline_chunk['S1_lg_mdot_f'], np.nan))

    DF_XRBs['Donor surf_avg_omega_div_omega_crit'] = np.where(mask1, oneline_chunk['S2_surf_avg_omega_div_omega_crit_f'],
                                                              np.where(mask2,oneline_chunk['S1_surf_avg_omega_div_omega_crit_f'],
                                                                       np.nan))

    DF_XRBs['Donor H1 surface content'] = np.where(mask1, oneline_chunk['S2_surface_h1_f'], 
                                                   np.where(mask2, oneline_chunk['S1_surface_h1_f'], np.nan))

    #===== Additional parameters related to the system =====
    DF_XRBs['Orbital period'] = oneline_chunk['orbital_period_f'].to_numpy()  # days
    DF_XRBs['Eccentricity'] = oneline_chunk['eccentricity_f'] 
    DF_XRBs['Log(mtransfer_rate)'] = oneline_chunk['lg_mtransfer_rate_f'] # Msun/yr
    DF_XRBs['metallicity'] = oneline_chunk['metallicity']

    #===== Calculate XLF for the population =====
    cols = DF_XRBs[['Donor mass', 'Donor log(R)', 'Accretor mass','Accretor log(R)', 'Accretor state', 'Log(mtransfer_rate)', 
                    'Donor log(mdot)', 'Accretor log(mdot)', 'Orbital period', 'Accretor spin', 
                    'Donor H1 surface content']].itertuples(index=False, name=None)
    
    # Store valid results 
    results = []

    # keep track of errors
    error_count = 0
    error_indices = []

    # Indices of the DataFrame
    idxs = DF_XRBs.index.to_numpy()

    # Luminosity calculation loop
    for i, row in enumerate(cols):
        Dm, Dlr, Am, Alr,Atype, log_mt, Dlog, Alog, Op, spin, h1 = row
        # skip invalid accretor/donor pairs early
        if Atype is None or (not np.isfinite(Dm) and not np.isfinite(Am)):
            results.append((np.nan, np.nan, np.nan))
            continue
        try:
            res = Compute_Lx(
                donor_mass = Dm,
                donor_log_R = Dlr,
                accretor_mass = Am,
                accretor_radius = Alr,
                accretor_type = Atype,
                log_Mtransfer_rate = log_mt,
                donor_log_mdot = Dlog,
                accretor_log_mdot = Alog,
                orbital_period_days = Op,
                accretor_spin = spin,
                H1_content_donor = h1,
                **kwargs
            )
        except Exception as e:
            # no logging available: record a minimal summary and continue
            error_count += 1
            # store the original DataFrame index for later inspection
            try:
                error_indices.append(idxs[i])
            except Exception:
                pass
            results.append((np.nan, np.nan, np.nan))
            continue

        # normalize result to tuple of three floats
        if res is None:
            results.append((np.nan, np.nan, np.nan))
        else:
            try:
                # Expect (Lx_total, Lx_RLO, Lx_BHL) or similar; normalize to (total, rlo, bhl)
                if len(res) == 3:
                    results.append(tuple(float(x) if np.isfinite(x) else np.nan for x in res))
                else:
                    # fallback: put total in first, others nan
                    results.append((float(res), np.nan, np.nan))
            except Exception:
                results.append((np.nan, np.nan, np.nan))

    if error_count:
        # succinct error display (no logging available)
        print(f"Compute_Lx: {error_count} error(s) in the luminosity calculation (Indices : {error_indices[:10]})")

    Lx_Total, Lx_RLO, Lx_BHL = zip(*results) if results else ([], [], [])
    
    DF_XRBs['Lx_RLO'] = Lx_RLO
    DF_XRBs['Lx_BHL'] = Lx_BHL
    DF_XRBs['Lx_Total'] = Lx_Total

    return DF_XRBs
